# Текущие static-low модели: SNO v2, FNO, Chebyshev

Все три модели используют одну фиксированную среду, k∈[1,2] и одинаковые 1280 test-поля. Notebook только читает готовые checkpoints; обучения нет. Полные test-метрики берутся из сохранённых metrics.json, а один заранее фиксированный test-пример пересчитывается для Re/Im/Abs. Сохранённых execution outputs нет.

[Отчёт и происхождение](../docs/operator_revision.md) · [Конфигурации и hashes](../docs/operator_revision.json).


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'neural_operator').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import json
import numpy as np
import matplotlib.pyplot as plt
from neural_operator.data import load_dataset
from neural_operator.training import load_predictor, dataset_fingerprint
from neural_operator.plotting import display_table, plot_prediction
BASE = ROOT / 'neural_operator'
DATA_DIR = BASE / 'data' / 'static_low_frequency_1000x8'
MODEL_RUNS = {
    'svd': BASE / 'runs' / 'revision_2026-11-08' / 'static_low' / 'svd',
    'chebyshev': BASE / 'runs' / 'revision_2026-11-08' / 'static_low' / 'chebyshev',
    'fno': BASE / 'randomized' / 'runs' / 'low_frequency_static_1000x8' / 'fno',
}


In [ ]:
dataset = load_dataset(DATA_DIR, mmap=True)
fingerprint = dataset_fingerprint(dataset)
test = dataset.split_indices()[2]
assert len(test) == 1280
rows = []
for method, directory in MODEL_RUNS.items():
    metrics = json.loads((directory / 'metrics.json').read_text())
    assert metrics['dataset_fingerprint'] == fingerprint
    assert metrics['test_samples'] == len(test)
    error = metrics['test_fem_l2']
    rows.append({'method': method, **{name+'_%': 100*value for name, value in error.items()},
                 'epochs': metrics['completed_epochs'], 'best_epoch': metrics['best_epoch'],
                 'parameters': metrics['parameters'], 'training_device': metrics['config']['device']})
display_table(rows)


## Re / Im / Abs одного и того же поля

Берём первый test-индекс; правило не использует ошибку или вид предсказания. Строки каждого рисунка — Re/Im/Abs; столбцы — FEM, модель, абсолютная ошибка компоненты. Комплексная FEM L2 дополнительно учитывает фазу. Это отдельный пример, а не замена метрик полного test.


In [ ]:
index = int(test[0])
print('sample:', index, 'k:', float(dataset.k0[index]), 'theta:', float(dataset.theta[index]))
for method, directory in MODEL_RUNS.items():
    predictor = load_predictor(directory / 'model.pt', device='cpu')
    prediction = predictor.predict_dataset(dataset, [index])[0]
    error = float(dataset.relative_l2(dataset.u_scat[index], prediction))
    print(method, 'sample complex FEM L2:', error)
    figure = plot_prediction(dataset, index, prediction, title=f'{method}: static-low, sample {index}')
    plt.show()
    plt.close(figure)


SNO и Chebyshev обучены на CPU, FNO — на MPS и переносится на CPU для inference. Различаются число параметров, завершённые эпохи и выходные пространства. Результаты относятся к этой среде и дискретному FEM reference; новые материалы и диапазоны требуют собственного исследования.
